# 07. Auditoría de Tags Editoriales y Propuesta de Normalización

Analizamos las etiquetas reales (`post_tag`) de los articulos de Anfibia, extraídas de la base de datos original.
Comparamos las etiquetas  frente a los **tópicos semánticos latentes** (BERTopic) para diagnosticar problemas de cobertura y solapamiento.

In [ ]:
# 1. Parche de compatibilidad obligatorio
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from bertopic import BERTopic
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos
df = pd.read_csv(DATA_PROCESSED)
df["doc_id"] = df.index

# 5. Carga de modelo y predicción
topic_model = BERTopic.load(MODEL_PATH)
topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()

print(f"Total de notas cruzadas: {len(df):,}")

Total de notas cruzadas: 3,556


## 1. Cobertura Histórica de Tags

In [56]:
df.columns

Index(['doc_id', 'ID', 'source_url', 'title_text', 'tagline', 'autor_names',
       'post_date', 'category', 'text_type', 'document_text_final',
       'document_text_original', 'post_tag'],
      dtype='str')

In [2]:
total = len(df)
df["tiene_tag"] = df["post_tag"].notna() & (df["post_tag"].str.strip() != "")

con_tag = df["tiene_tag"].sum()
sin_tag = total - con_tag

print(f"Notas con etiquetas: {con_tag:,} ({(con_tag/total)*100:.1f}%)")
print(f"Notas SIN etiquetas: {sin_tag:,} ({(sin_tag/total)*100:.1f}%)")

fig = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
fig.update_layout(title="Cobertura de Etiquetas en el Archivo", width=500, height=400)
fig.show()

Notas con etiquetas: 1,589 (44.7%)
Notas SIN etiquetas: 1,967 (55.3%)


In [ ]:
# Agrupamos por año: Total de notas vs Notas con tag
evolucion = (
    df.groupby("post_year")
    .agg(
        total_notas=("doc_id", "count"),
        notas_con_tag=("tiene_tag", "sum")
    )
    .reset_index()
)
evolucion["notas_sin_tag"] = evolucion["total_notas"] - evolucion["notas_con_tag"]
evolucion["pct_con_tag"] = (evolucion["notas_con_tag"] / evolucion["total_notas"] * 100).round(1)
evolucion["pct_sin_tag"] = (evolucion["notas_sin_tag"] / evolucion["total_notas"] * 100).round(1)

# Filtramos posibles años nulos o erróneos
evolucion = evolucion[evolucion["post_year"] > 2010]

# 3. Gráfico de barras apiladas
fig_evo = go.Figure()

# Texto condicional: oculta el porcentaje si la barra es menor al 6% para evitar desbordes
textos_verdes = [f"{p}%" if p > 6 else "" for p in evolucion["pct_con_tag"]]

# Barra inferior: Notas que SÍ tienen tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_con_tag"],
    name="Con Etiquetas",
    marker_color='#2ca02c',
    text=textos_verdes,
    textposition='inside',
    textangle=0,  # <--- ACÁ ESTÁ EL CAMBIO: 0 para texto horizontal
    insidetextanchor='middle',
    customdata=evolucion["pct_con_tag"],
    hovertemplate="<b>%{x}</b><br>Con etiquetas: %{y} notas (%{customdata}%)<extra></extra>"
))

# Barra superior: Notas SIN tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_sin_tag"],
    name="Sin Etiquetas (Vacío)",
    marker_color='#d62728',
    customdata=evolucion["pct_sin_tag"],
    hovertemplate="<b>%{x}</b><br>Sin etiquetas: %{y} notas (%{customdata}%)<extra></extra>"
))

fig_evo.update_layout(
    barmode='stack',
    title="<b>Notas con y sin Tags por Año</b>",
    xaxis_title="Año de Publicación",
    yaxis_title="Cantidad de Artículos",
    height=500,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig_evo.update_xaxes(type='category')
fig_evo.write_html(OUTPUT_DIR / "03_evolucion_etiquetado.html")
fig_evo.show()

## 2. Fragmentación y Uso Único de Etiquetas
Separamos las etiquetas múltiples (divididas por `;`) y analizamos su frecuencia.

In [9]:
TOP = 15
# Desanidar tags separados por punto y coma
tags_list = (
    df["post_tag"]
    .dropna()
    .str.split(r";")
    .explode()
    .str.strip()
    .loc[lambda x: x != ""]
)

tag_counts = tags_list.value_counts()
tags_unicos = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()

print(f"Cantidad de tags únicos creados: {tags_unicos:,}")
print(f"Tags usados en 1 sola nota: {tags_una_vez:,} ({(tags_una_vez/tags_unicos)*100:.1f}%)")

print(f"\nTop {TOP} Tags más usados:")
display(tag_counts.head(TOP).to_frame("Frecuencia"))

Cantidad de tags únicos creados: 881
Tags usados en 1 sola nota: 444 (50.4%)

Top 15 Tags más usados:


,Frecuencia
post_tag,
Internacional,238
Javier Milei,162
Derechas,114
Zapping cultural,84
Derechos Humanos,77
UNSAM,67
Chile,65
Feminismos,62
Peronismo,61


In [47]:
# ==============================================================================
# 2.3. Muestreo y Diagnóstico de Tags Unicos
# ==============================================================================

# 1. Asegurar la serie de tags limpios
tags_series = df["post_tag"].dropna().str.split(r"[;,]").explode().str.strip()
tags_series = tags_series[tags_series != ""]

# 2. Filtrar tags que aparecen exactamente 1 sola vez
counts = tags_series.value_counts()
unicos = counts[counts == 1].index.to_series().reset_index(drop=True)

df_diag = pd.DataFrame({"tag": unicos})
df_diag["cant_palabras"] = df_diag["tag"].str.split().str.len()

# 3. Distribución por cantidad de palabras
print("Distribución de longitud (palabras por tag unico):")
for n_words, total in df_diag["cant_palabras"].value_counts().sort_index().items():
    pct = (total / len(df_diag)) * 100
    print(f" • {n_words} palabra(s): {total} tags ({pct:.1f}%)")

print("\n" + "=" * 60)
print(f"Muestra aleatoria de 30 tags Unicos (de un total de {len(df_diag)}):")
print("=" * 60)

# 4. Muestra representativa
muestra = df_diag["tag"].sample(n=min(30, len(df_diag)), random_state=42).tolist()
for i, t in enumerate(muestra, 1):
    print(f"{i:2d}. {t}")

Distribución de longitud (palabras por tag unico):
 • 1 palabra(s): 232 tags (52.3%)
 • 2 palabra(s): 159 tags (35.8%)
 • 3 palabra(s): 39 tags (8.8%)
 • 4 palabra(s): 11 tags (2.5%)
 • 5 palabra(s): 3 tags (0.7%)

Muestra aleatoria de 30 tags Unicos (de un total de 444):
 1. Nora Dalmasso
 2. cancelación
 3. Honduras
 4. Canal de Panamá
 5. Xiomara Castro
 6. Autista
 7. Familiares
 8. Noche
 9. Pep
10. Imaginación política
11. Ciudades
12. Amistad
13. #EAAF
14. Inteligencia
15. Ecología
16. gendarmería
17. Derechos sociales
18. Disidencias sexuales
19. Moda
20. Beca crónica
21. Randazzo
22. Animales
23. Palito Ortega
24. Liliana Herrero
25. Instagram
26. Vínculos
27. Invasión
28. Ríos entubados
29. Fundamentalistas del Aire Acondicionado
30. Soledad Acuña


## 3. Etiquetas en las categorias

Recordamos como se distribuyen las categorias.

In [ ]:
## 2.1 Distribución de Categorías Editoriales
import re

# 1. Normalizar y explotar manteniendo el dataframe listo
df_cat_exploded = (
    df[["ID", "post_year", "category", "post_tag"]]
    .copy()
    .assign(
        categoria_normalizada=lambda d: d["category"]
        .fillna("Sin categoria")
        .astype(str)
        .apply(lambda x: [p.strip() for p in re.split(r"[;,]", x) if p.strip()] or ["Sin categoria"])
    )
    .explode("categoria_normalizada")
    .reset_index(drop=True)
)

# 2. Conteo de artículos por categoría
df_cat_dist = (
    df_cat_exploded.groupby("categoria_normalizada")["ID"]
    .nunique()
    .reset_index(name="notas")
    .sort_values("notas", ascending=True)
)

# 3. Gráfico breve
fig_cat = px.bar(
    df_cat_dist, x="notas", y="categoria_normalizada", orientation="h",
    title="<b>Distribución de Notas por Categoría (Universo Lematizado)</b>",
    text="notas", color="notas", color_continuous_scale="Blues"
)
fig_cat.update_traces(textposition="outside")
fig_cat.update_layout(
    height=400, plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="#fff",
    coloraxis_showscale=False, xaxis_title="", yaxis_title="",
    margin=dict(l=150, r=40, t=50, b=30)
)
fig_cat.write_html(OUTPUT_DIR / "03a_distribucion_categorias.html")
fig_cat.show()

Mostramos como se distribuyen N-TOP etiquetas sobre las categorias.

In [34]:
TOP = 15

# 1. Desanidar los tags directamente sobre df_cat_exploded
df_tags_cat = df_cat_exploded.dropna(subset=["post_tag"]).copy()
df_tags_cat["tag"] = df_tags_cat["post_tag"].str.split(r"[;,]")
df_tags_cat = df_tags_cat.explode("tag").reset_index(drop=True)
df_tags_cat["tag"] = df_tags_cat["tag"].str.strip().str.title()
df_tags_cat = df_tags_cat[df_tags_cat["tag"] != ""]

# 2. Top N global y cálculo de totales por tag (artículos únicos)
tag_counts = df_tags_cat.groupby("tag")["ID"].nunique().sort_values(ascending=False).head(TOP)
top_nombres = tag_counts.index.tolist()

mapeo_etiqueta_total = {tag: f"{tag} ({conteo})" for tag, conteo in tag_counts.items()}
orden_filas = [mapeo_etiqueta_total[tag] for tag in top_nombres][::-1]

# 3. Filtrar y formatear
df_top = df_tags_cat[df_tags_cat["tag"].isin(top_nombres)].reset_index(drop=True)
df_top["tag_label"] = df_top["tag"].map(mapeo_etiqueta_total)

# 4. Matriz de Frecuencia (Tag vs Categoría Normalizada)
matriz = pd.crosstab(df_top["tag_label"], df_top["categoria_normalizada"])
matriz = matriz.reindex(orden_filas)
matriz = matriz.loc[:, matriz.sum(axis=0) > 0]

# 5. Render del Heatmap
fig_heatmap = px.imshow(
    matriz,
    labels=dict(x="Categoría Editorial", y="", color="Notas"),
    x=matriz.columns,
    y=matriz.index,
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=True
)

fig_heatmap.update_traces(
    hovertemplate="<b>Tag:</b> %{y}<br><b>Categoría:</b> %{x}<br><b>Notas en sección:</b> %{z}<extra></extra>"
)

fig_heatmap.update_layout(
    title=f"<b>Top {TOP} Etiquetas (Total de Notas) vs Categorías Editoriales</b>",
    xaxis_title="",
    yaxis_title="",
    height=max(500, TOP * 32),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(l=180, b=100)
)
fig_heatmap.update_xaxes(tickangle=-45)

fig_heatmap.write_html(OUTPUT_DIR / f"03b_top{TOP}_tags_heatmap.html")
fig_heatmap.show()

In [49]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np

# ==============================================================================
# 2.3. Donas por Categoría con Dropdown Temporal
# ==============================================================================

TOP_TAGS_POR_CAT = 10  # <--- Cambiá este número para ver el Top 3, Top 10, etc.
MIN_NOTAS_CAT = 20

# 1. Categorías válidas basadas en artículos únicos totales
notas_totales_por_cat = df_cat_exploded[df_cat_exploded["categoria_normalizada"] != "Sin categoria"].groupby("categoria_normalizada")["ID"].nunique()
categorias_validas = notas_totales_por_cat[notas_totales_por_cat >= MIN_NOTAS_CAT].sort_values(ascending=False).index.tolist()

años_disponibles = sorted([int(a) for a in df_cat_exploded["post_year"].dropna().unique() if a > 2010])
pasos_temporales = ["Global"] + [str(a) for a in años_disponibles]

n_cats = len(categorias_validas)
n_cols = 3
n_rows = int(np.ceil(n_cats / n_cols))

# Títulos limpios, solo el nombre de la categoría
titulos_iniciales = [f"<b>{cat}</b>" for cat in categorias_validas]

fig_donuts = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "domain"}] * n_cols for _ in range(n_rows)],
    subplot_titles=titulos_iniciales,
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

colores_anillo = px.colors.qualitative.Set2

def formatear_label(txt, max_len=14):
    if len(txt) > max_len:
        partes = txt.split(" ")
        if len(partes) > 1:
            medio = len(partes) // 2
            return " ".join(partes[:medio]) + "<br>" + " ".join(partes[medio:])
    return txt

# 2. Generar trazas para cada período
for paso_idx, paso in enumerate(pasos_temporales):
    es_global = (paso == "Global")
    
    if es_global:
        df_tags_periodo = df_tags_cat
    else:
        df_tags_periodo = df_tags_cat[df_tags_cat["post_year"] == int(paso)]

    es_visible_inicial = (paso_idx == 0)

    for idx, cat in enumerate(categorias_validas):
        r = (idx // n_cols) + 1
        c = (idx % n_cols) + 1

        df_c = df_tags_periodo[df_tags_periodo["categoria_normalizada"] == cat]
        top_tags = df_c["tag"].value_counts().head(TOP_TAGS_POR_CAT)

        if len(top_tags) > 0:
            labels_originales = top_tags.index.tolist()
            values = top_tags.values.tolist()
            labels_formateados = [formatear_label(l) for l in labels_originales]
            colores = colores_anillo[:len(labels_formateados)]
        else:
            labels_originales = ["Sin etiquetas en el período"]
            values = [1]
            labels_formateados = ["Sin tags"]
            colores = ["#f0f0f0"]

        fig_donuts.add_trace(
            go.Pie(
                labels=labels_formateados,
                customdata=labels_originales,
                values=values,
                hole=0.60,
                name=f"{cat} ({paso})",
                marker=dict(colors=colores, line=dict(color='#ffffff', width=2)),
                textinfo="label+percent" if len(top_tags) > 0 else "none",
                textposition="outside",
                hovertemplate="<b>%{customdata}</b><br>Artículos con este tag: %{value} (%{percent})<extra></extra>" if len(top_tags) > 0 else "<b>Sin tags registrados</b><extra></extra>",
                showlegend=False,
                visible=es_visible_inicial
            ),
            row=r,
            col=c
        )

# 3. Botones del Dropdown
total_trazas = len(fig_donuts.data)
trazas_por_paso = n_cats

dropdown_buttons = []
for i, paso in enumerate(pasos_temporales):
    visibilidad = [False] * total_trazas
    inicio = i * trazas_por_paso
    fin = inicio + trazas_por_paso
    visibilidad[inicio:fin] = [True] * trazas_por_paso

    dropdown_buttons.append(dict(
        method="update",
        args=[
            {"visible": visibilidad},
            {"title": f"<b>Distribución de Tags por Categoría (Top {TOP_TAGS_POR_CAT}) — Período: {paso}</b>"}
        ],
        label=f"  {paso}  "
    ))

updatemenus = [dict(
    buttons=dropdown_buttons,
    direction="down",
    pad={"r": 10, "t": 10},
    showactive=True,
    x=0.0,
    xanchor="left",
    y=1.12,
    yanchor="top",
    bgcolor="#ffffff",
    bordercolor="#cccccc",
    font=dict(size=12, color="#333333")
)]

fig_donuts.update_layout(
    updatemenus=updatemenus,
    title_text=f"<b>Distribución de Tags por Categoría (Top {TOP_TAGS_POR_CAT}) — Período: Global</b>",
    height=480 * n_rows + 40,
    width=1650,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(t=120, b=40, l=60, r=60),
    font=dict(size=11)
)

fig_donuts.write_html(OUTPUT_DIR / "03c_tags_donas_dropdown_temporal.html")
fig_donuts.show()

## 3. Solapamiento: Tags vs Tópicos BERTopic
Vemos cómo un mismo tópico semántico fue etiquetado con múltiples tags manuales distintos.

In [5]:
# Armamos df desanidado completo para agrupar
df_desanidado = df.drop("post_tag", axis=1).join(
    df["post_tag"].dropna().str.split(r";").explode().str.strip().rename("tag")
).dropna(subset=["tag"])

df_desanidado = df_desanidado[df_desanidado["tag"] != ""]

# Agrupamos por tópico
solapamiento = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby(["topic", "topic_name"])
    .agg(
        notas_con_tag=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_tags=("tag", lambda x: " | ".join(pd.Series.unique(x)[:5]))
    )
    .reset_index()
    .sort_values("notas_con_tag", ascending=False)
)

display(solapamiento.head(10))
solapamiento.to_csv(OUTPUT_DIR / "auditoria_tags_por_topico.csv", index=False)

,topic,topic_name,notas_con_tag,tags_unicos_usados,ejemplos_tags
0,0,"político, política, electoral",113,144,Escuela de Política y Gobierno | UNSAM | Brexi...
1,1,"feminismo, feminista, patriarcado",63,66,#NiUnaMenos | LGBTIQ+ | Orgullo | Memoria tran...
2,2,"fifa, futbolista, fútbol",50,54,Perfil político | Barras bravas | Boca Juniors...
4,4,"déficit, deuda, endeudamiento",43,46,Fondos buitre | Movimientos sociales | Violenc...
5,5,"pedagógico, enseñanza, escolar",38,51,Encuentro Anfibio | Feria del Libro | Día del ...
18,18,"chileno, chile, político",36,27,Diego Maradona | Chile | Estallido en Chile | ...
13,13,"papal, catolicismo, arzobispo",32,29,Papa Francisco | Rossana Reguillo | Catolicism...
10,10,"boliviano, bolivia, político",30,43,Catolicismo | Papa Francisco | Paraguay | Reli...
16,16,"brasil, brasilero, brasilia",30,16,Brasil | Dilma Rousseff | Internacional | Mund...
8,8,"epidemia, vacunar, virus",23,24,Rosariocronica | Rosario | Covid-19 | Escuela ...


Cruzemos las Notas vs la Cantidad de Tags Únicos. Y ademas calculemos un "Índice de Fragmentación" (cuántos tags se crean por cada nota) que sirva como una tercera dimensión (color): 

si la burbuja es roja, significa que los editores inventaron casi un tag nuevo por cada artículo de ese tema.

In [6]:
import plotly.express as px

MOSTRAR_POLITICA = False

# 1. Calculamos el ratio de fragmentación sobre solapamiento
solapamiento["ratio_fragmentacion"] = (solapamiento["tags_unicos_usados"] / solapamiento["notas_con_tag"]).round(2)

# 2. Filtramos tópicos con volumen suficiente y aplicamos la flag para el Tópico 0
df_ranking = solapamiento[solapamiento["notas_con_tag"] >= 15].copy()

if not MOSTRAR_POLITICA:
    df_ranking = df_ranking[df_ranking["topic"] != 0]

# 3. Ordenamos por ratio de fragmentación (ascendente para que el peor quede arriba en barras horizontales)
df_ranking = df_ranking.sort_values("ratio_fragmentacion", ascending=True).tail(15)

# 4. Bar Chart Horizontal
fig_ranking = px.bar(
    df_ranking,
    x="ratio_fragmentacion",
    y="topic_name",
    orientation="h",
    color="ratio_fragmentacion",
    color_continuous_scale="Reds",
    text="tags_unicos_usados",
    hover_data={
        "topic": True,
        "notas_con_tag": True,
        "tags_unicos_usados": True,
        "ratio_fragmentacion": False,
        "ejemplos_tags": True,
    },
    title="<b>Proporción de etiquetas por artículo</b><br><sup>El número en la barra indica la cantidad total de tags únicos inventados para ese tema.</sup>",
    labels={
        "ratio_fragmentacion": "Índice de Caos (Tags únicos / Notas)",
        "topic_name": "Palabras del Tópico",
    },
    height=650,
    template="plotly_white",
)

# Ajustes de visualización y margen
fig_ranking.update_traces(textposition="outside")
fig_ranking.update_layout(
    xaxis_range=[0, df_ranking["ratio_fragmentacion"].max() * 1.15],
    coloraxis_showscale=False,
)

# 5. Exportar y mostrar
fig_ranking.write_html(OUTPUT_DIR / "05_ranking_caos_etiquetas.html")
fig_ranking.show()

## 4. Ideas de recomendacíon
